In [1]:
# Run cells selectively according to your needs.
from tqdm.auto import tqdm

from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

In [ ]:
import os
from pathlib import Path
import datasets
import multiprocess as mp

DISKROOT = "/FS1"

hf_cache_dir = os.path.join(DISKROOT, "datasets/hf_datasets_cache")
dataset_path = os.path.join(DISKROOT, "datasets/OpenWebText")
os.makedirs(hf_cache_dir, exist_ok=True)
tokenized_dataset_path = dataset_path + '_tokenized'
chunked_tokenized_dataset_path = tokenized_dataset_path + '_chunked'

def tokenize(batch, tokenizer):
    return tokenizer(
        batch["text"], 
        truncation=True,        # Truncates texts longer than max_length
        max_length=1024         # Explicitly set to match your model's context
    )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_DATASETS_CACHE"] = os.path.join(DISKROOT, "datasets/hf_datasets_cache")
num_proc = 7
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass


In [ ]:
# 5. Import and load with explicit cache_dir override
from datasets import load_dataset
dataset = load_dataset(
    dataset_path, 
    split='train',
    cache_dir=hf_cache_dir  # Explicitly forces data writing to /FS1
)


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [ ]:
print(f'Starting tokenizing with num_proc={num_proc}')
token_ids = dataset.map(
    tokenize,
    batched=True,
    batch_size=1000,
    num_proc=num_proc,
    remove_columns=dataset.column_names,
    fn_kwargs={"tokenizer": tokenizer}  # <--- Serializes and sends tokenizer to workers
)

# 5. Persist final dataset
print('Tokenization completes. Saving to disk...')
token_ids.save_to_disk(tokenized_dataset_path)


Starting tokenizing with num_proc=7


Map (num_proc=7):   0%|          | 0/8013769 [00:00<?, ? examples/s]

Tokenization completes. Saving to disk...


Saving the dataset (0/56 shards):   0%|          | 0/8013769 [00:00<?, ? examples/s]

In [6]:
token_ids = load_from_disk(tokenized_dataset_path)

Loading dataset from disk:   0%|          | 0/56 [00:00<?, ?it/s]

In [13]:
eos_id= tokenizer.eos_token_id

def group_texts(examples, eos_id):
    # Append eos_id to each document before concatenating
    block_size = 1024
    concatenated_ids = []
    for seq in examples["input_ids"]:
        concatenated_ids.extend(seq)
        concatenated_ids.append(eos_id)

    total_length = len(concatenated_ids)
    total_length = (total_length // block_size) * block_size

    return {
        "input_ids": [
            concatenated_ids[i : i + block_size]
            for i in range(0, total_length, block_size)
        ]
    }

chunked_dataset = token_ids.map(
    group_texts,
    batched=True,
    num_proc=num_proc, # Now num_proc works here
    desc="Grouping texts",
    remove_columns=token_ids.column_names,
    fn_kwargs={"eos_id": eos_id}
)
chunked_dataset.save_to_disk(chunked_tokenized_dataset_path)

Grouping texts (num_proc=7):   0%|          | 0/8013769 [00:00<?, ? examples/s]

Saving the dataset (0/45 shards):   0%|          | 0/5458075 [00:00<?, ? examples/s]

In [ ]:
chunked_dataset = load_from_disk(chunked_tokenized_dataset_path)

In [15]:
# list dataset splits
chunked_dataset

Dataset({
    features: ['input_ids'],
    num_rows: 5458075
})